### Middleware
Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:

Tracking agent behavior with logging, analytics, and debugging.
Transforming prompts, tool selection, and output formatting.
Adding retries, fallbacks, and early termination logic.
Applying rate limits, guardrails, and PII detection.

In [14]:
import os
from dotenv import load_dotenv
load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

### Summarization MiddleWare

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:

1. Long-running conversations that exceed context windows.
2. Multi-turn dialogues with extensive history.
3. Applications where preserving full conversation context matters.

In [15]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [16]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain.messages import SystemMessage, HumanMessage

agent = create_agent(
    model="groq:qwen/qwen3.6-27b",
    checkpointer=InMemorySaver(),

    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.6-27b",
            trigger=("messages", 10),
            keep=("messages", 4),
        )
    ]
)

In [17]:
# run with thread_id
config = {"configurable" : {"thread_id": "test-1"}}

In [18]:
# Alternative test data
questions = [
    "What is 2+2?",
    "What is 10*5?",
    "What is 100/4?",
    "What is 15-7?",
    "What is 3*3?",
    "What is 4*4?",
]

for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response['messages'])}")

Messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='d75b352c-5f80-41bb-8dcb-b7eba4d8f8fa'), AIMessage(content='\n<think>\nHere\'s a thinking process:\n\n1.  **Analyze User Input:** The user asks "What is 2+2?"\n2.  **Identify Core Task:** Simple arithmetic addition.\n3.  **Perform Calculation:** 2 + 2 = 4.\n4.  **Formulate Response:** State the answer clearly and concisely.\n5.  **Check for Complexity/Traps:** None. It\'s a straightforward math question.\n6.  **Output Generation:** "2 + 2 equals 4." (or simply "4")\n\nI\'ll go with a clear, direct answer.✅\n</think>\n\n2 + 2 equals **4**.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 144, 'prompt_tokens': 17, 'total_tokens': 161, 'completion_time': 0.27552234, 'completion_tokens_details': None, 'prompt_time': 0.000296836, 'prompt_tokens_details': None, 'queue_time': 0.053753184, 'total_time': 0.275819176}, 'model_name': 'qwen/qwen3.6-27b', 'syste

### Token Size

In [19]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel - 5 star, $350/night, spa, pool, gym
    2. City Inn - 4 star, $180/night, business center
    3. Budget Stay - 3 star, $75/night, free wifi"""


agent=create_agent(
    model="groq:qwen/qwen3.6-27b",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3.6-27b",
            trigger=("tokens",550),
            keep=("tokens",200),
        ),
    ]
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter (approximate)
def count_tokens(messages):
    total_chars = sum(len(str(m.content)) for m in messages)
    return total_chars // 4  # 4 chars ≈ 1 token

In [20]:
# Run test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )
    
    tokens = count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response['messages'])} messages")
    print(f"{(response['messages'])}")

Paris: ~125 tokens, 4 messages
[HumanMessage(content='Find hotels in Paris', additional_kwargs={}, response_metadata={}, id='844f3fbc-1849-4ff3-b2c2-efa56e827026'), AIMessage(content='', additional_kwargs={'reasoning_content': 'Thinking Process:\n1.  **Identify User Intent**: The user wants to find hotels in Paris.\n2.  **Identify Available Tool**: `search_hotels` function is available.\n3.  **Check Parameters**: `search_hotels` requires `city` (string). The user provided "Paris".\n4.  **Execute Tool Call**: Call `search_hotels(city="Paris")`.\n5.  **Process Response**: The tool will return a list of hotels. I will format and present them to the user. I\'ll wait for the tool output.👇🏼\n', 'tool_calls': [{'id': 'q62f1j0k9', 'function': {'arguments': '{"city":"Paris"}', 'name': 'search_hotels'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 157, 'prompt_tokens': 277, 'total_tokens': 434, 'completion_time': 0.297665446, 'completion_tokens_details': {'reaso

### Fraction

In [22]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.checkpoint.memory import InMemorySaver

@tool
def search_hotels(city: str) -> str:
    """Search hotels."""
    return f"Hotels in {city}: Grand Hotel $350, City Inn $180, Budget Stay $75"

# LOW fraction for testing!
agent = create_agent(
    model="gpt-4o-mini",
    tools=[search_hotels],
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="gpt-4o-mini",
            trigger=("fraction", 0.005),  # 0.5% = ~640 tokens
            keep=("fraction", 0.002),     # 0.2% = ~256 tokens
        ),
    ],
)

config = {"configurable": {"thread_id": "test-1"}}

# Token counter
def count_tokens(messages):
    return sum(len(str(m.content)) for m in messages) // 4

# Test
cities = ["Paris", "London", "Tokyo", "New York", "Dubai", "Singapore"]

for city in cities:
    response = agent.invoke(
        {"messages": [HumanMessage(content=f"Hotels in {city}")]},
        config=config
    )
    tokens = count_tokens(response["messages"])
    fraction = tokens / 128000  # gpt-4o-mini context
    print(f"{city}: ~{tokens} tokens ({fraction:.4%}), {len(response['messages'])} msgs")
    print(response['messages'])

Paris: ~71 tokens (0.0555%), 4 msgs
[HumanMessage(content='Hotels in Paris', additional_kwargs={}, response_metadata={}, id='7d24fe29-11b9-4d79-8255-ed7f54286f7a'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 44, 'total_tokens': 59, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_9709ac0e8b', 'id': 'chatcmpl-EAaVjHrvhn1Jl64RNJze3Dicv9NqJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019fe15c-0e9f-7343-a42c-41556ff19195-0', tool_calls=[{'name': 'search_hotels', 'args': {'city': 'Paris'}, 'id': 'call_aszIVqsYM7Q7jfZJQTJL8HwU', 'type': 'tool_call'}], invalid_tool_calls=[], usage_